# Importação das bibliotecas necessárias

In [6]:
import sys
import os
sys.path.append('/home/jovyan/work')

from utils.AlinharETL import AlinharETL

# Crie uma instância da classe AlinharETL

In [7]:
# Crie uma instância da classe
bucket = "silver"
datamart = "compliance"
extrator_bronze = AlinharETL(bucket,datamart)

2024-10-10 15:15:02,494 - INFO - Iniciando Sessão Spark.


# Leitura das tabelas

In [8]:
df_caged = extrator_bronze.criar_view_temporaria('silver/NovoCaged/MicrodadosNovoCaged','compliance_caged')

2024-10-10 15:15:03,216 - INFO - Aguarde... Criando View (silver/NovoCaged/MicrodadosNovoCaged)
2024-10-10 15:15:09,478 - INFO - View criada com sucesso


In [9]:
df_cnae = extrator_bronze.criar_view_temporaria('silver/ServicoDadosIBGE/Cnae','compliance_cnae')

2024-10-10 15:15:09,485 - INFO - Aguarde... Criando View (silver/ServicoDadosIBGE/Cnae)
2024-10-10 15:15:09,880 - INFO - View criada com sucesso


In [10]:
df_cbo = extrator_bronze.criar_view_temporaria('gold/MTE/FatoMTECbo','compliance_cbo')

2024-10-10 15:15:09,885 - INFO - Aguarde... Criando View (gold/MTE/FatoMTECbo)
2024-10-10 15:15:10,283 - INFO - View criada com sucesso


In [11]:
df_caged_municipio = extrator_bronze.criar_view_temporaria('silver/NovoCaged/MicrodadosNovoCagedMunicio','compliance_caged_municipio') 

2024-10-10 15:15:10,290 - INFO - Aguarde... Criando View (silver/NovoCaged/MicrodadosNovoCagedMunicio)
2024-10-10 15:15:10,739 - INFO - View criada com sucesso


In [12]:
df_caged_secao = extrator_bronze.criar_view_temporaria('silver/NovoCaged/MicrodadosNovoCagedSecao','compliance_caged_secao') 

2024-10-10 15:15:10,745 - INFO - Aguarde... Criando View (silver/NovoCaged/MicrodadosNovoCagedSecao)
2024-10-10 15:15:11,083 - INFO - View criada com sucesso


In [13]:
df_caged_subclasse = extrator_bronze.criar_view_temporaria('silver/NovoCaged/MicrodadosNovoCagedSubClasse','compliance_caged_subclasse') 

2024-10-10 15:15:11,089 - INFO - Aguarde... Criando View (silver/NovoCaged/MicrodadosNovoCagedSubClasse)
2024-10-10 15:15:11,383 - INFO - View criada com sucesso


In [14]:
df_caged_cbo2002ocupacao = extrator_bronze.criar_view_temporaria('silver/NovoCaged/MicrodadosNovoCagedCbo2002Ocupacao','compliance_caged_cbo2002ocupacao') 

2024-10-10 15:15:11,389 - INFO - Aguarde... Criando View (silver/NovoCaged/MicrodadosNovoCagedCbo2002Ocupacao)
2024-10-10 15:15:11,684 - INFO - View criada com sucesso


In [15]:
df_caged_tipomovimentacao = extrator_bronze.criar_view_temporaria('silver/NovoCaged/MicrodadosNovoCagedTipoMovimentacao','compliance_caged_tipomovimentacao') 

2024-10-10 15:15:11,690 - INFO - Aguarde... Criando View (silver/NovoCaged/MicrodadosNovoCagedTipoMovimentacao)
2024-10-10 15:15:11,973 - INFO - View criada com sucesso


In [16]:
df_caged_indicadoraprendiz = extrator_bronze.criar_view_temporaria('silver/NovoCaged/MicrodadosNovoCagedIndicadorAprendiz','compliance_caged_indicadoraprendiz') 

2024-10-10 15:15:11,979 - INFO - Aguarde... Criando View (silver/NovoCaged/MicrodadosNovoCagedIndicadorAprendiz)
2024-10-10 15:15:12,246 - INFO - View criada com sucesso


# Tratamentos na tabela 

In [18]:
sql_query = """
    SELECT
        to_date(concat_ws('-', substring(caged.dsc_competencia_mov, 1, 4), substring(caged.dsc_competencia_mov, 5, 2), '01'), 'yyyy-MM-dd') AS dsc_competencia_mov,
        compliance_caged_municipio.descricao AS dsc_municipio,
        compliance_caged_secao.descricao AS dsc_secao,
        compliance_caged_subclasse.descricao AS dsc_subclasse,
        caged.dsc_saldo_movimentacao,
        compliance_caged_cbo2002ocupacao.descricao AS dsc_cbo_ocupacao,
        caged.vl_horas_contratuais,
        compliance_caged_tipomovimentacao.descricao AS dsc_tipo_movimentacao,
        caged.vl_salario,
        compliance_caged_indicadoraprendiz.descricao AS dsc_indicador_aprendiz,
        caged.vl_salario_fixo,
        compliance_cnae.dsc_divisao,
        compliance_cnae.dsc_grupo,
        compliance_cnae.dsc_classe, 
        compliance_cbo.dsc_grande_grupo,
        compliance_cbo.dsc_subgrupo_principal,
        compliance_cbo.dsc_subgrupo,
        compliance_cbo.demanda_formacao as dsc_demanda_formacao,
        compliance_cbo.dsc_industrial_uniepro
    FROM compliance_caged caged
    JOIN compliance_caged_municipio ON caged.cd_municipio = compliance_caged_municipio.codigo 
    JOIN compliance_caged_secao ON caged.cd_secao = compliance_caged_secao.codigo 
    JOIN compliance_caged_subclasse ON caged.cd_subclasse = compliance_caged_subclasse.codigo 
    JOIN compliance_caged_cbo2002ocupacao ON caged.cd_cbo_ocupacao = compliance_caged_cbo2002ocupacao.codigo 
    JOIN compliance_caged_tipomovimentacao ON caged.cd_tipo_movimentacao = compliance_caged_tipomovimentacao.codigo 
    JOIN compliance_caged_indicadoraprendiz ON caged.cd_indicador_aprendiz = compliance_caged_indicadoraprendiz.codigo 
    JOIN compliance_cnae ON caged.cd_subclasse = compliance_cnae.sk_cnae
    JOIN compliance_cbo ON caged.cd_cbo_ocupacao = compliance_cbo.cd
"""

df_compliance_cbo_grande_sub_grupo = extrator_bronze.carregar_dados_delta(sql_query)

2024-10-10 15:15:19,554 - INFO - Aguarde... Executando Query (Delta)
2024-10-10 15:15:19,556 - INFO - 
    SELECT
        to_date(concat_ws('-', substring(caged.dsc_competencia_mov, 1, 4), substring(caged.dsc_competencia_mov, 5, 2), '01'), 'yyyy-MM-dd') AS dsc_competencia_mov,
        compliance_caged_municipio.descricao AS dsc_municipio,
        compliance_caged_secao.descricao AS dsc_secao,
        compliance_caged_subclasse.descricao AS dsc_subclasse,
        caged.dsc_saldo_movimentacao,
        compliance_caged_cbo2002ocupacao.descricao AS dsc_cbo_ocupacao,
        caged.vl_horas_contratuais,
        compliance_caged_tipomovimentacao.descricao AS dsc_tipo_movimentacao,
        caged.vl_salario,
        compliance_caged_indicadoraprendiz.descricao AS dsc_indicador_aprendiz,
        caged.vl_salario_fixo,
        compliance_cnae.dsc_divisao,
        compliance_cnae.dsc_grupo,
        compliance_cnae.dsc_classe, 
        compliance_cbo.dsc_grande_grupo,
        compliance_cbo.dsc_sub

In [19]:
df_compliance_cbo_grande_sub_grupo.count()

1554646

# Gravação no datalake

In [13]:
extrator_bronze.caminho_tabela_delta = 's3a://gold/NovoCaged/FlatMicrodadosNovoCaged'

In [14]:
extrator_bronze.salvar_delta(df_compliance_cbo_grande_sub_grupo, 'overwrite')

2024-09-25 19:24:21,487 - INFO - Aguarde... Persistindo dados (overwrite)
2024-09-25 19:25:00,102 - INFO - Dados persistidos com sucesso
2024-09-25 19:25:00,103 - INFO - s3a://gold/NovoCaged/FlatMicrodadosNovoCaged
2024-09-25 19:25:00,103 - INFO - Aguarde... Realizando optimize
2024-09-25 19:25:01,496 - INFO - Optimize executado com sucesso.
2024-09-25 19:25:01,497 - INFO - Aguarde... Realizando vacuum
2024-09-25 19:25:14,156 - INFO - Vacuum executado com sucesso.


# Encerra sessão spark

In [15]:
extrator_bronze.parar_sessao()

2024-09-25 19:25:14,742 - INFO - Sessão Spark finalizada.
